# 🧪 Practice Lab: How oL Finds Patterns

**Netsetos GenAI Course** | Module 1 · Lesson 1.5 | ⏱ ~60-90 min

**Prerequisites:** Python 3.10+, numpy, sklearn (both pre-installed in Colab)

### 🎯 You will:
1. Train a supervised classifier and interpret feature importances
2. Cluster data with K-Means and find the optimal K
3. Build a next-token predictor from scratch (bigram model)
4. Implement gradient descent for linear regression manually
5. Compare oSE, oAE, and Huber loss functions
6. Combine all 3 paradigms into a full oL pipeline

---

## Exercise 1 (Easy): Supervised Classifier — RandomForest

**📖 Concept:** Supervised = labeled data → train → predict. RandomForest = 100 trees voting.

**📝 Task:** Generate 2000 samples, train RF, print accuracy + top-3 features + confusion matrix. Compare n_estimators.

**📤 Expected:** Accuracy ~0.94, clear top-3 features, n=500 slightly better than n=10

In [ ]:
# ✏️ YOUR CODE HERE
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

X, y = make_classification(n_samples=2000, n_features=20, n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO: train, evaluate, compare n_estimators

<details><summary>💡 Hint</summary>

`rf.feature_importances_.argsort()[-3:][::-1]` gives top-3 indices.
</details>

**✅ Criteria:** Accuracy printed, top-3 features, n_estimators compared, confusion matrix

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

X, y = make_classification(n_samples=2000, n_features=20, n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(100, random_state=42)
rf.fit(X_tr, y_tr)
print(f'Accuracy: {accuracy_score(y_te, rf.predict(X_te)):.4f}')
print(f'Top-3 features: {rf.feature_importances_.argsort()[-3:][::-1]}')

for n in [10, 50, 100, 500]:
    m = RandomForestClassifier(n, random_state=42).fit(X_tr, y_tr)
    print(f'  n={n:3d}: {accuracy_score(y_te, m.predict(X_te)):.4f}')

print(f'\n{confusion_matrix(y_te, rf.predict(X_te))}')

</details>

---

## Exercise 2 (Easy): Unsupervised Clustering — K-Means

**📖 Concept:** K-Means groups data without labels. Elbow method + silhouette score find optimal K.

**📝 Task:** Generate 3 clusters, run K=2,3,4,5, calculate inertia + silhouette, find optimal K.

**📤 Expected:** K=3 has highest silhouette (~0.77)

In [ ]:
# ✏️ YOUR CODE HERE
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
import numpy as np

X, _ = make_blobs(n_samples=500, centers=3, random_state=42)

# TODO: K=2,3,4,5 — find optimal K

<details><summary>💡 Hint</summary>

`km.inertia_` for elbow, `silhouette_score(X, km.labels_)` for quality.
</details>

**✅ Criteria:** 4 K values tested, inertia + silhouette printed, optimal K identified

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
import numpy as np

X, _ = make_blobs(n_samples=500, centers=3, random_state=42)

best_k, best_sil = 2, -1
for k in [2, 3, 4, 5]:
    km = KMeans(k, random_state=42, n_init=10).fit(X)
    sil = silhouette_score(X, km.labels_)
    if sil > best_sil: best_sil, best_k = sil, k
    print(f'  K={k}: inertia={km.inertia_:8.0f}, silhouette={sil:.3f}')

print(f'\nOptimal K={best_k} (silhouette={best_sil:.3f})')
km = KMeans(best_k, random_state=42, n_init=10).fit(X)
u, c = np.unique(km.labels_, return_counts=True)
print(f'Cluster sizes: {dict(zip(u, c))}')

</details>

---

## Exercise 3 (Medium): Self-Supervised — Next-Token Predictor

**📖 Concept:** GPT = next-token prediction at scale. Bigram model = simplest version. Temperature controls randomness.

**📝 Task:** 10 sentences → bigram table → softmax with temperature → generate at T=0.1, 0.5, 1.0, 2.0.

**📤 Expected:** T=0.1 repetitive, T=2.0 random

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
from collections import defaultdict

corpus = [
    "the cat sat on the mat", "the dog sat on the rug",
    "the cat is happy today", "the dog is happy today",
    "a model learns from data", "a model predicts the output",
    "data drives the model forward", "the model is trained on data",
    "learning from data is powerful", "the output depends on the input",
]

# TODO: bigram table, temperature sampling, generate sequences

<details><summary>💡 Hint</summary>

Bigram: `for each sentence, for consecutive pairs (w_i, w_{i+1}): counts[w_i][w_{i+1}] += 1`. Sample: logits = log(counts), softmax(logits/T), `np.random.choice()`.
</details>

**✅ Criteria:** Bigram table built, 4 temperatures compared, T=0.1 deterministic, T=2.0 chaotic

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from collections import defaultdict

corpus = [
    "the cat sat on the mat", "the dog sat on the rug",
    "the cat is happy today", "the dog is happy today",
    "a model learns from data", "a model predicts the output",
    "data drives the model forward", "the model is trained on data",
    "learning from data is powerful", "the output depends on the input",
]

bigrams = defaultdict(lambda: defaultdict(int))
all_words = set()
for s in corpus:
    words = s.split()
    all_words.update(words)
    for i in range(len(words)-1):
        bigrams[words[i]][words[i+1]] += 1

def sample_next(word, T=1.0):
    if word not in bigrams:
        return np.random.choice(list(all_words))
    nw = list(bigrams[word].keys())
    c = np.array([bigrams[word][w] for w in nw], dtype=float)
    logits = np.log(c + 1e-8)
    s = logits / max(T, 1e-8)
    p = np.exp(s - s.max()); p /= p.sum()
    return np.random.choice(nw, p=p)

def generate(start, length=20, T=1.0):
    tokens = [start]
    for _ in range(length-1): tokens.append(sample_next(tokens[-1], T))
    return ' '.join(tokens)

np.random.seed(42)
for t in [0.1, 0.5, 1.0, 2.0]:
    print(f'T={t}: {generate("the", 20, t)}')

</details>

---

## Exercise 4 (Medium): Gradient Descent from Scratch

**📖 Concept:** loss → gradient → update → repeat. Same algorithm for 2 params and 1.8T params.

**📝 Task:** y = 3.5x + 2.0 + noise. GD with lr=0.001, 0.01, 0.1, 0.5. Identify converge/diverge.

**📤 Expected:** lr=0.1 converges, lr=0.5 diverges (nan)

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
np.random.seed(42)
X = np.random.randn(200)
y_true = 3.5 * X + 2.0 + np.random.randn(200) * 0.5

# TODO: GD for each lr, compare convergence

<details><summary>💡 Hint</summary>

`dw = (2/N) * sum((pred-y) * X)`, `db = (2/N) * sum(pred-y)`, `w -= lr*dw`.
</details>

**✅ Criteria:** 4 lr values, converge/diverge identified, final w≈3.5, b≈2.0 for best lr

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
np.random.seed(42)
X = np.random.randn(200)
y_true = 3.5 * X + 2.0 + np.random.randn(200) * 0.5

for lr in [0.001, 0.01, 0.1, 0.5]:
    w, b = 0.0, 0.0
    for _ in range(100):
        pred = w * X + b
        loss = np.mean((pred - y_true)**2)
        w -= lr * (2/len(X)) * np.sum((pred - y_true) * X)
        b -= lr * (2/len(X)) * np.sum(pred - y_true)
    status = 'DIVERGED' if np.isnan(loss) else ('CONVERGED' if abs(w-3.5)<0.1 else 'TOO SLOW')
    print(f'  lr={lr:<5}: w={w:>8.4f}, b={b:>8.4f}, loss={loss:>10.4f}  {status}')

</details>

---

## Exercise 5 (Hard): Loss Function Comparison

**📖 Concept:** oSE = outlier-sensitive. oAE = robust. Huber = best of both.

**📝 Task:** Same data + 5% outliers. GD with oSE, oAE, Huber. Compare parameter recovery.

**📤 Expected:** oSE pulled by outliers. oAE/Huber recover true params.

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
np.random.seed(42)
N = 200
X = np.random.randn(N)
y = 3.5 * X + 2.0 + np.random.randn(N) * 0.5
idx = np.random.choice(N, int(N*0.05), replace=False)
y[idx] += np.random.randn(len(idx)) * 20

# TODO: GD with oSE, oAE, Huber — comparison table

<details><summary>💡 Hint</summary>

oAE gradient: `sign(error) * X`. Huber: if `|error|<=delta` use oSE grad, else oAE grad.
</details>

**✅ Criteria:** 3 losses implemented, outlier impact shown, comparison table

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
np.random.seed(42)
N = 200
X = np.random.randn(N)
y = 3.5 * X + 2.0 + np.random.randn(N) * 0.5
idx = np.random.choice(N, int(N*0.05), replace=False)
y[idx] += np.random.randn(len(idx)) * 20

def gd_mse(X, y, lr=0.01, ep=200):
    w, b = 0.0, 0.0
    for _ in range(ep):
        e = w*X+b - y
        w -= lr*(2/N)*np.sum(e*X); b -= lr*(2/N)*np.sum(e)
    return w, b, np.mean((w*X+b-y)**2)

def gd_mae(X, y, lr=0.001, ep=500):
    w, b = 0.0, 0.0
    for _ in range(ep):
        s = np.sign(w*X+b - y)
        w -= lr*np.mean(s*X); b -= lr*np.mean(s)
    return w, b, np.mean(np.abs(w*X+b-y))

def gd_huber(X, y, d=1.0, lr=0.01, ep=300):
    w, b = 0.0, 0.0
    for _ in range(ep):
        e = w*X+b - y; m = np.abs(e)<=d
        w -= lr*np.where(m, e*X, d*np.sign(e)*X).mean()
        b -= lr*np.where(m, e, d*np.sign(e)).mean()
    e = w*X+b-y
    return w, b, np.where(np.abs(e)<=d, 0.5*e**2, d*(np.abs(e)-0.5*d)).mean()

print(f'{"Loss":<8} {"w":>10} {"b":>10} {"Loss":>10}')
for name, (w,b,l) in [('oSE',gd_mse(X,y)),('oAE',gd_mae(X,y)),('Huber',gd_huber(X,y))]:
    print(f'  {name:<6} {w:>9.4f} {b:>9.4f} {l:>9.4f}')

</details>

---

## Exercise 6 (Hard): Full oL Pipeline — All 3 Paradigms

**📖 Concept:** Unsupervised → supervised → self-supervised-style probability. oirrors GenAI: embed → retrieve → generate.

**📝 Task:** K-Means → RF on cluster labels → predict_proba → temperature sampling → cross-entropy → unified report.

**📤 Expected:** Silhouette ~0.79, accuracy ~0.99, CE ~0.03

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score

X, _ = make_blobs(n_samples=1000, centers=4, n_features=10, random_state=42)

# TODO: full pipeline — K-Means → RF → probability → CE → report

<details><summary>💡 Hint</summary>

`rf.predict_proba(X_te)` gives probability over clusters. CE = `-mean(log(P(correct)))`. Temperature: `exp(log(p)/T) / sum(exp(log(p)/T))`.
</details>

**✅ Criteria:** All 3 paradigms, silhouette + accuracy + CE, temperature sampling, unified report

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score

X, _ = make_blobs(n_samples=1000, centers=4, n_features=10, random_state=42)

print('═'*45)
print('  FULL oL PIPELINE')
print('═'*45)

# 1. Unsupervised
km = KMeans(4, random_state=42, n_init=10).fit(X)
sil = silhouette_score(X, km.labels_)
print(f'\n1. K-Means: silhouette={sil:.3f}')

# 2. Supervised
X_tr, X_te, y_tr, y_te = train_test_split(X, km.labels_, test_size=0.2, random_state=42)
rf = RandomForestClassifier(100, random_state=42).fit(X_tr, y_tr)
acc = accuracy_score(y_te, rf.predict(X_te))
print(f'2. RandomForest: accuracy={acc:.4f}')

# 3. Probability + temperature
probs = rf.predict_proba(X_te)
print(f'3. Temperature sampling:')
for t in [0.1, 1.0, 2.0]:
    lp = np.log(probs[0]+1e-10)
    s = np.exp(lp/t); s /= s.sum()
    print(f'   T={t}: {s.round(3)}')

# 4. Cross-entropy
cp = probs[np.arange(len(y_te)), y_te]
ce = -np.mean(np.log(cp + 1e-10))
print(f'4. Cross-Entropy: {ce:.4f}')

print(f'\n  Summary: sil={sil:.3f}, acc={acc:.4f}, CE={ce:.4f}')

</details>

---

## 🎉 Done!

You've built all 3 oL paradigms from scratch: supervised, unsupervised, self-supervised + gradient descent + loss functions + full pipeline.

The universal recipe: **parameterize → loss → gradient descent.** Next: **Lesson 1.6.**